# LAB 01 — From Text Processing to Search

Parts A–C are in calculations.md and prediction.md. This notebook follows Parts D–J from W1.pdf.

In [1]:
# Google Colab setup: clone the repository and use it as the working directory.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/thangSy221105/NLP_exercise.git"
REPO_DIR = Path("/content/NLP_exercise")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        print(f"Repository already exists: {REPO_DIR}")
    os.chdir(REPO_DIR)
    print(f"Working directory: {Path.cwd()}")
else:
    print("Colab clone cell skipped outside Google Colab.")

Working directory: /content/NLP_exercise


# 7. Part D — Experiment 1: Inspect the Sparse Representation

## 7.1. Dataset

Use the provided corpus of approximately 30,000 documents. Do not change the dataset for this experiment.

The repository contains c4-train.00000-of-01024-30K.json.gz under lab01/data/. It is a gzip-compressed JSON Lines file with a text field.

In [36]:
import gzip
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("lab01/data/c4-train.00000-of-01024-30K.json.gz")
DATA_AVAILABLE = False
documents = []
document_ids = []
records = []

if not DATA_PATH.is_file():
    print(f"Dataset not found at {DATA_PATH}.")
else:
    with gzip.open(DATA_PATH, "rt", encoding="utf-8") as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    documents = [record["text"] for record in records]
    document_ids = list(range(len(documents)))
    DATA_AVAILABLE = True
    print(f"Loaded {len(documents):,} documents from {DATA_PATH}.")
    print(f"Empty documents: {sum(not text.strip() for text in documents):,}")
    display(pd.DataFrame(records[:3]))

Loaded 30,000 documents from lab01/data/c4-train.00000-of-01024-30K.json.gz.
Empty documents: 0


,text,timestamp,url
0,Beginners BBQ Class Taking Place in Missoula!\...,2019-04-25T12:57:54Z,https://klyq.com/beginners-bbq-class-taking-pl...
1,Discussion in 'Mac OS X Lion (10.7)' started b...,2019-04-21T10:07:13Z,https://forums.macrumors.com/threads/restore-f...
2,Foil plaid lycra and spandex shortall with met...,2019-04-25T10:40:23Z,https://awishcometrue.com/Catalogs/Clearance/T...


## 7.2. Xây dựng pipeline

Raw documents
->
Tokenizer
->
CountVectorizer
->
TF
->
IDF
->
TF-IDF matrix


In [37]:
if DATA_AVAILABLE:
    try:
        from scipy import sparse
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.preprocessing import normalize
        SKLEARN_AVAILABLE = True
    except ImportError as error:
        SKLEARN_AVAILABLE = False
        print(f"Experiment dependencies are unavailable: {error}")
else:
    SKLEARN_AVAILABLE = False

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)
    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    number_of_documents = count_matrix.shape[0]
    idf_values = np.log(number_of_documents / document_frequency)
    tfidf_matrix = tf_matrix.multiply(idf_values).tocsr()
    tfidf_matrix.eliminate_zeros()
    feature_names = vectorizer.get_feature_names_out()
else:
    print("Pipeline skipped because the dataset is not configured.")

## 7.3. Kiểm tra kích thước

In [38]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    print(f"Number of documents = {count_matrix.shape[0]}")
    print(f"Vocabulary size = {count_matrix.shape[1]}")
    print(f"Matrix shape = {tfidf_matrix.shape}")
else:
    print("Size check skipped because the dataset is not configured.")

Number of documents = 30000
Vocabulary size = 193540
Matrix shape = (30000, 193540)


> N = 30000
>
> V = 193540

## 7.4. Kiểm tra sparsity


In [ ]:
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    nnz = tfidf_matrix.nnz
    total_entries = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
    sparsity = 1 - nnz / total_entries if total_entries else 0.0
    print(f"nnz = {nnz}")
    print(f"Total entries = {total_entries}")
    print(f"Sparsity = {sparsity}")
    print(f"Count matrix sparse = {sparse.issparse(count_matrix)}")
    print(f"TF matrix sparse = {sparse.issparse(tf_matrix)}")
    print(f"TF-IDF matrix sparse = {sparse.issparse(tfidf_matrix)}")
else:
    print("Sparsity check skipped because the dataset is not configured.")

nnz = 4985822
Total entries = 5806200000
Sparsity = 0.9991412934449382
Count matrix sparse = True
TF matrix sparse = True
TF-IDF matrix sparse = True


**Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V?**

>Mặc dù một document chỉ chứa một phần nhỏ các term trong vocabulary, vector của nó vẫn có chiều \(V\) vì mỗi term trong vocabulary được gán với một vị trí cố định trong vector. Term xuất hiện trong document sẽ có giá trị khác 0, còn term không xuất hiện sẽ có giá trị 0. Nhờ đó, tất cả document đều được biểu diễn trong cùng một không gian \(V\) chiều và có thể so sánh với nhau.

## 7.5. Inspect vocabulary

### 7.5.1. Top 20 terms theo document frequency

In [ ]:
def get_top_df_terms(document_frequency, feature_names, top_k=20):
    table = pd.DataFrame({"term": feature_names, "document_frequency": document_frequency})
    return table.sort_values("document_frequency", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    display(get_top_df_terms(document_frequency, feature_names))
else:
    print("Document-frequency inspection skipped because the dataset is not configured.")

,term,document_frequency
0,the,27893
1,and,27423
2,to,26689
3,of,26031
4,in,25224
5,for,23651
6,is,22739
7,with,21405
8,on,20262
9,that,18370


### 7.5.2. Top 20 terms có IDF cao nhất

In [ ]:
def get_top_idf_terms(feature_names, idf_values, top_k=20):
    table = pd.DataFrame({"term": feature_names, "idf": idf_values})
    return table.sort_values("idf", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    display(get_top_idf_terms(feature_names, idf_values))
else:
    print("IDF inspection skipped because the dataset is not configured.")

,term,idf
0,𐌼𐌿𐌽𐌳𐍃,10.308953
1,ﬂuid,10.308953
2,훈장,10.308953
3,후에는,10.308953
4,효과를,10.308953
5,회화적인,10.308953
6,회화의,10.308953
7,회화성의,10.308953
8,회화성을,10.308953
9,회화사의,10.308953


### 7.5.3. Top 20 terms có TF-IDF cao nhất trong một document được chọn

In [ ]:
def get_top_tfidf_terms(tfidf_matrix, feature_names, document_index, top_k=20):
    row = tfidf_matrix.getrow(document_index)
    table = pd.DataFrame({"term": feature_names[row.indices], "tfidf": row.data})
    return table.sort_values("tfidf", ascending=False).head(top_k).reset_index(drop=True)

SELECTED_DOC_INDEX = 0
if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    print(f"Selected document preview: {documents[SELECTED_DOC_INDEX][:500]}")
    display(get_top_tfidf_terms(tfidf_matrix, feature_names, SELECTED_DOC_INDEX))
else:
    print("Document TF-IDF inspection skipped because the dataset is not configured.")

Selected document preview: Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat select


,term,tfidf
0,bbq,0.184033
1,class,0.097049
2,balay,0.081173
3,kcbs,0.075715
4,meat,0.074776
5,lonestar,0.072522
6,missoula,0.063042
7,apron,0.058414
8,smoker,0.057988
9,timelines,0.054134


### 7.5.4. So sánh ba danh sách

1. Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?
>Không. Nếu một term xuất hiện trong rất nhiều document thì document frequency (DF) cao, dẫn đến IDF thấp. Vì TF-IDF = TF × IDF, nên dù term xuất hiện nhiều, TF-IDF của nó vẫn có thể thấp. Ví dụ: từ 'the' xuất hiện rất nhiều trong corpus, nhưng vì nó xuất hiện ở hầu hết document nên không giúp phân biệt document nào với document nào → IDF thấp → TF-IDF thường thấp.

2. Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?
>Không. IDF cao chỉ cho biết term đó hiếm trong toàn corpus. Muốn TF-IDF cao trong một document cụ thể thì term đó còn phải xuất hiện trong document đó và có TF đủ lớn. Nếu term không xuất hiện thì TF = 0 → TF-IDF = 0, dù IDF có cao đến đâu

# 8. Part E — Core Implementation

## 8.2. Các hàm cần xây dựng

- build_vocabulary()
- compute_counts()
- compute_tf()
- compute_idf()
- compute_tfidf()
- cosine_similarity()

In [6]:
import math
from collections import Counter

documents = [
    "cat eats fish",
    "dog eats fish",
    "cat likes fish"
]


# 1. Xây dựng vocabulary
def build_vocabulary(documents):
    vocab = set()

    for doc in documents:
        tokens = doc.split()
        vocab.update(tokens)

    return sorted(vocab)


# 2. Đếm số lần mỗi term xuất hiện trong document
def compute_counts(document, vocabulary):
    tokens = document.split()
    counter = Counter(tokens)

    return [counter[word] for word in vocabulary]


# 3. Tính TF
def compute_tf(document, vocabulary):
    tokens = document.split()
    total_terms = len(tokens)
    counter = Counter(tokens)

    return [
        counter[word] / total_terms
        for word in vocabulary
    ]


# 4. Tính IDF
def compute_idf(documents, vocabulary):
    N = len(documents)
    idf = []

    for word in vocabulary:
        df = sum(
            1 for doc in documents
            if word in doc.split()
        )

        idf.append(math.log(N / df))

    return idf


# 5. Tính TF-IDF
def compute_tfidf(document, vocabulary, idf):
    tf = compute_tf(document, vocabulary)

    return [
        tf[i] * idf[i]
        for i in range(len(vocabulary))
    ]


# 6. Cosine similarity
def cosine_similarity(x, y):
    dot_product = sum(a * b for a, b in zip(x, y))

    norm_x = math.sqrt(sum(a * a for a in x))
    norm_y = math.sqrt(sum(b * b for b in y))

    if norm_x == 0 or norm_y == 0:
        return 0.0

    return dot_product / (norm_x * norm_y)

## 8.3. Corpus kiểm thử

- D1 = cat eats fish
- D2 = dog eats fish
- D3 = cat likes fish

In [7]:
vocabulary = build_vocabulary(documents)

assert vocabulary == [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]

print(vocabulary)

['cat', 'dog', 'eats', 'fish', 'likes']


## 8.4. Unit tests

In [10]:
# =========================
# UNIT TESTS
# =========================

documents = [
    "cat eats fish",
    "dog eats fish",
    "cat likes fish"
]

# 1. Test build_vocabulary()
vocabulary = build_vocabulary(documents)

print("1. Vocabulary:")
print(vocabulary)

assert vocabulary == [
    "cat",
    "dog",
    "eats",
    "fish",
    "likes"
]


# 2. Test compute_counts()
counts_d1 = compute_counts(
    documents[0],
    vocabulary
)

print("\n2. Count vector of D1:")
print(counts_d1)

assert counts_d1 == [1, 0, 1, 1, 0]


# 3. Test compute_tf()
tf_d1 = compute_tf(
    documents[0],
    vocabulary
)

print("\n3. TF of D1:")
print(tf_d1)

tf_cat = tf_d1[
    vocabulary.index("cat")
]

assert abs(
    tf_cat - (1 / 3)
) < 1e-9


# 4. Test compute_idf()
idf = compute_idf(
    documents,
    vocabulary
)

print("\n4. IDF:")
for term, value in zip(vocabulary, idf):
    print(f"{term}: {value}")

idf_cat = idf[
    vocabulary.index("cat")
]

assert abs(
    idf_cat - math.log(3 / 2)
) < 1e-9


# 5. Test compute_tfidf()
tfidf_d1 = compute_tfidf(
    documents[0],
    vocabulary,
    idf
)

print("\n5. TF-IDF of D1:")
for term, value in zip(vocabulary, tfidf_d1):
    print(f"{term}: {value}")

tfidf_cat = tfidf_d1[
    vocabulary.index("cat")
]

assert abs(
    tfidf_cat -
    ((1 / 3) * math.log(3 / 2))
) < 1e-9


# 6. Test cosine_similarity()
x = [1, 1, 1]
y = [1, 1, 0]

similarity = cosine_similarity(
    x,
    y
)

print("\n6. Cosine similarity:")
print(similarity)

assert abs(
    similarity -
    (2 / math.sqrt(6))
) < 1e-9

1. Vocabulary:
['cat', 'dog', 'eats', 'fish', 'likes']

2. Count vector of D1:
[1, 0, 1, 1, 0]

3. TF of D1:
[0.3333333333333333, 0.0, 0.3333333333333333, 0.3333333333333333, 0.0]

4. IDF:
cat: 0.4054651081081644
dog: 1.0986122886681098
eats: 0.4054651081081644
fish: 0.0
likes: 1.0986122886681098

5. TF-IDF of D1:
cat: 0.13515503603605478
dog: 0.0
eats: 0.13515503603605478
fish: 0.0
likes: 0.0

6. Cosine similarity:
0.8164965809277259


## 8.5. So sánh với thư viện

In [11]:
# =========================
# 8.5. COMPARE WITH LIBRARY
# =========================

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

documents = [
    "cat eats fish",
    "dog eats fish",
    "cat likes fish"
]

# Vocabulary từ student implementation
vocabulary = build_vocabulary(documents)

# Giữ đúng thứ tự vocabulary để dễ so sánh
vocab_map = {
    term: i
    for i, term in enumerate(vocabulary)
}

print("Vocabulary:")
print(vocabulary)


# ---------------------------------
# 1. Count Vector - Library
# ---------------------------------

count_vectorizer = CountVectorizer(
    vocabulary=vocab_map
)

X_count = count_vectorizer.fit_transform(
    documents
).toarray()

print("\n1. Count vectors - Library:")

for i, vector in enumerate(X_count):
    print(f"D{i+1}: {vector}")


# ---------------------------------
# 2. IDF + TF-IDF - Library
# ---------------------------------

tfidf_vectorizer = TfidfVectorizer(
    vocabulary=vocab_map,
    smooth_idf=False,
    norm=None
)

X_tfidf = tfidf_vectorizer.fit_transform(
    documents
).toarray()

print("\n2. IDF - Library:")

for term, value in zip(
    vocabulary,
    tfidf_vectorizer.idf_
):
    print(f"{term}: {value}")


print("\n3. TF-IDF vectors - Library:")

for i, vector in enumerate(X_tfidf):
    print(f"D{i+1}: {vector}")


# ---------------------------------
# 3. Student implementation
# ---------------------------------

student_idf = compute_idf(
    documents,
    vocabulary
)

print("\n4. IDF - Student:")

for term, value in zip(
    vocabulary,
    student_idf
):
    print(f"{term}: {value}")


print("\n5. TF-IDF vectors - Student:")

student_tfidf = []

for i, document in enumerate(documents):

    vector = compute_tfidf(
        document,
        vocabulary,
        student_idf
    )

    student_tfidf.append(vector)

    print(f"D{i+1}: {vector}")


# ---------------------------------
# 4. Compare cosine similarity
# ---------------------------------

student_similarity = cosine_similarity(
    student_tfidf[0],
    student_tfidf[1]
)

library_similarity = sklearn_cosine_similarity(
    X_tfidf[0].reshape(1, -1),
    X_tfidf[1].reshape(1, -1)
)[0][0]


print("\n6. Cosine similarity D1 - D2")

print(
    "Student implementation:",
    student_similarity
)

print(
    "Library:",
    library_similarity
)

Vocabulary:
['cat', 'dog', 'eats', 'fish', 'likes']

1. Count vectors - Library:
D1: [1 0 1 1 0]
D2: [0 1 1 1 0]
D3: [1 0 0 1 1]

2. IDF - Library:
cat: 1.4054651081081644
dog: 2.09861228866811
eats: 1.4054651081081644
fish: 1.0
likes: 2.09861228866811

3. TF-IDF vectors - Library:
D1: [1.40546511 0.         1.40546511 1.         0.        ]
D2: [0.         2.09861229 1.40546511 1.         0.        ]
D3: [1.40546511 0.         0.         1.         2.09861229]

4. IDF - Student:
cat: 0.4054651081081644
dog: 1.0986122886681098
eats: 0.4054651081081644
fish: 0.0
likes: 1.0986122886681098

5. TF-IDF vectors - Student:
D1: [0.13515503603605478, 0.0, 0.13515503603605478, 0.0, 0.0]
D2: [0.0, 0.3662040962227032, 0.13515503603605478, 0.0, 0.0]
D3: [0.13515503603605478, 0.0, 0.0, 0.0, 0.3662040962227032]

6. Cosine similarity D1 - D2
Student implementation: 0.24482975009584623
Library: 0.49225493709608925


> Student implementation và reference implementation có cùng vocabulary và count vectors, nhưng IDF và TF-IDF có numerical results khác nhau. Nguyên nhân là Student implementation sử dụng (TF=count/total\_terms\) và (IDF=log(N/df)\), trong khi TfidfVectorizer với smooth_idf=False sử dụng raw term frequency và \(IDF=log(N/df)+1\). Vì vậy, term fish xuất hiện trong tất cả documents có IDF bằng 0 trong Student implementation nhưng bằng 1 trong library. Sự khác biệt này cũng làm cosine similarity giữa D1 và D2 thay đổi. Đây là sự khác nhau về formula/convention, không cho thấy Student implementation bị sai.

# 9. Part F — Experiment 2: Preprocessing Ablation

## 9.1. Pipeline A — Minimal

Raw text
->
Lowercase
->
Tokenization

In [12]:
import re

def tokenize_with_punctuation(text: str) -> list[str]:
    return re.findall(r"\b\w+\b|[^\w\s]", str(text).lower(), flags=re.UNICODE)

def preprocess_a(text: str) -> list[str]:
    return tokenize_with_punctuation(text)

## 9.2. Pipeline B — Normalized

Raw text
->
Lowercase
->
Punctuation normalization
->
Tokenization
->
Stopword handling

In [13]:
STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "it", "of", "on", "or", "that", "the", "this", "to", "was", "were", "with"}

def tokenize_words(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", str(text).lower(), flags=re.UNICODE)

def preprocess_b(text: str) -> list[str]:
    return [token for token in tokenize_words(text) if token not in STOPWORDS]

## 9.3. Pipeline C — Extended

Raw text
->
Normalization
->
Subword tokenization


In [14]:
def preprocess_c(text: str) -> list[str]:
    subwords = []
    for word in tokenize_words(text):
        if len(word) <= 3:
            subwords.append(word)
        else:
            bounded_word = f"<{word}>"
            subwords.extend(bounded_word[i:i + 3] for i in range(len(bounded_word) - 2))
    return subwords

## 9.4. So sánh



In [44]:
# ============================================
# PART F - 9.4 PREPROCESSING ABLATION
# FULL COMPARISON TABLE
# ============================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity


pipeline_results = {}
pipeline_models = {}


# ============================================
# Function: build TF-IDF
# ============================================

def build_tfidf(processed_documents):

    vectorizer = CountVectorizer()

    count_matrix = vectorizer.fit_transform(
        processed_documents
    )


    # TF
    tf_matrix = normalize(
        count_matrix,
        norm="l1",
        axis=1
    )


    # IDF
    df = np.asarray(
        count_matrix.getnnz(axis=0)
    ).ravel()


    N = count_matrix.shape[0]


    idf = np.log(
        N / df
    )


    # TF-IDF
    tfidf_matrix = (
        tf_matrix.multiply(idf)
        .tocsr()
    )


    tfidf_matrix.eliminate_zeros()


    return (
        vectorizer,
        tfidf_matrix
    )



# ============================================
# Function: Search performance
# ============================================

def evaluate_search(
    vectorizer,
    matrix,
    preprocess,
    queries
):

    scores = []


    for query in queries:


        query_tokens = preprocess(query)


        query_text = " ".join(
            query_tokens
        )


        query_vector = vectorizer.transform(
            [query_text]
        )


        similarity = cosine_similarity(
            query_vector,
            matrix
        )[0]


        # Top 5 similarity average

        top5 = np.argsort(
            similarity
        )[-5:]


        scores.append(
            np.mean(
                similarity[top5]
            )
        )


    return np.mean(scores)

# ==========================
# Calculate OOV rate
# ==========================

def calculate_oov_rate(
    eval_documents,
    vocabulary,
    preprocess
):

    total_tokens = 0
    oov_tokens = 0

    vocab_set = set(vocabulary)


    for doc in eval_documents:

        tokens = preprocess(doc)


        for token in tokens:

            total_tokens += 1

            if token not in vocab_set:
                oov_tokens += 1


    return oov_tokens / max(total_tokens, 1)

# ============================================
# Run A/B/C
# ============================================


for name, preprocess in pipeline_specs.items():


    print("Running:", name)


    # ------------------------
    # Preprocessing
    # ------------------------

    processed_documents = []


    original_token_count = 0
    processed_token_count = 0


    for doc in documents:


        original_tokens = doc.split()


        tokens = preprocess(doc)


        original_token_count += len(
            original_tokens
        )

        processed_token_count += len(
            tokens
        )


        processed_documents.append(
            " ".join(tokens)
        )



    # ------------------------
    # TF-IDF
    # ------------------------

    vectorizer, matrix = build_tfidf(
        processed_documents
    )


    # ------------------------
    # Metrics
    # ------------------------

    vocab_size = len(
        vectorizer.get_feature_names_out()
    )


    avg_tokens = (
        processed_token_count
        /
        len(documents)
    )


    sparsity = (
        1 -
        matrix.nnz /
        (
            matrix.shape[0]
            *
            matrix.shape[1]
        )
    )


    # OOV rate

    oov_rate = calculate_oov_rate(
        documents,
        vectorizer.get_feature_names_out(),
        preprocess
    )


    # Search

    search_score = evaluate_search(
        vectorizer,
        matrix,
        preprocess,
        EXAMPLE_QUERIES
    )


    # save

    pipeline_models[name] = {

        "vectorizer":
            vectorizer,

        "matrix":
            matrix,

        "documents":
            processed_documents
    }


    pipeline_results[name] = {


        "Vocabulary size":
            vocab_size,


        "Average tokens/document":
            avg_tokens,


        "Matrix sparsity":
            sparsity,


        "OOV rate":
            oov_rate,


        "Search performance":
            search_score

    }


    print(
        name,
        "completed:",
        matrix.shape
    )



# ============================================
# FINAL COMPARISON TABLE
# ============================================

comparison_table = pd.DataFrame(
    pipeline_results
).T


comparison_table

Running: Pipeline A
Pipeline A completed: (30000, 193540)
Running: Pipeline B
Pipeline B completed: (30000, 193518)
Running: Pipeline C
Pipeline C completed: (30000, 41668)


,Vocabulary size,Average tokens/document,Matrix sparsity,OOV rate,Search performance
Pipeline A,193540.0,430.327033,0.999141,0.183554,0.299020
Pipeline B,193518.0,266.721900,0.999213,0.038307,0.299710
Pipeline C,41668.0,1526.351967,0.988180,0.300265,0.345824


## 9.5. Câu hỏi phân tích

1. Lowercasing làm thay đổi vocabulary như thế nào?

> Lowercasing giúp gộp các token khác nhau do chữ hoa/chữ thường về cùng một dạng. Trong experiment này, vocabulary giảm nhẹ từ 193540 xuống 193518, cho thấy tác động của lowercasing không lớn trên corpus.

2. Stopword removal có luôn cải thiện representation không?

> Không. Trong experiment này, Pipeline B giảm số token trung bình mỗi document (430.33 → 266.72) và tăng nhẹ search performance (0.299020 → 0.299710). Tuy nhiên, stopword removal không phải lúc nào cũng tốt vì một số bài toán có thể cần các từ chức năng.

3. Việc loại punctuation có thể làm mất thông tin gì?

> Loại punctuation giúp giảm sự khác biệt giữa các token như "word" và "word,". Tuy nhiên, nó có thể làm mất thông tin về cấu trúc câu hoặc biểu cảm trong một số bài toán NLP.


4. Pipeline nào tạo ra sparse matrix nhất?

> Pipeline B tạo ra sparse matrix nhất với sparsity = 0.999213, cao hơn Pipeline A (0.999141) và Pipeline C (0.988180). Việc loại bỏ các token ít quan trọng làm tăng số lượng giá trị 0 trong ma trận.

5. Pipeline nào cho search tốt nhất?

> Pipeline C cho kết quả search tốt nhất với score = 0.345824. Mặc dù có vocabulary nhỏ hơn nhiều (41668 so với khoảng 193000 của A/B), Pipeline C vẫn tạo ra sự tương đồng tốt hơn giữa query và document.

6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

> Không. Pipeline C có vocabulary nhỏ nhất nhưng đạt search performance cao nhất. Điều này cho thấy hiệu quả tìm kiếm còn phụ thuộc vào cách tokenization, preprocessing và biểu diễn dữ liệu, không chỉ phụ thuộc vào kích thước vocabulary.

# 10. Part G — Application: Build a Document Search Engine

## 10.1. Bài toán

Input:
User query

Output:
Top-K relevant documents

In [47]:
# =====================================
# Part G - Application
# G1. Build TF-IDF Index
# =====================================

# Sử dụng pipeline tốt nhất từ Part F
# (ở đây chọn Pipeline C)

search_pipeline = "Pipeline C"


search_vectorizer = pipeline_models[
    search_pipeline
]["vectorizer"]


search_matrix = pipeline_models[
    search_pipeline
]["matrix"]


search_documents = pipeline_models[
    search_pipeline
]["documents"]


print("Number of documents:",
      search_matrix.shape[0])

print("Vocabulary size:",
      search_matrix.shape[1])

Number of documents: 30000
Vocabulary size: 41668


## 10.2. Pipeline

30K Documents
↓
TF-IDF Index
↓
User Query
↓
TF-IDF Query Vector
↓
Cosine Similarity
↓
Ranking
↓
Top-K Documents

In [48]:
# =====================================
# G2. Query Processing and Ranking
# =====================================

def search_engine(query, top_k=5):


    # Query preprocessing

    preprocess = pipeline_specs[
        search_pipeline
    ]


    query_tokens = preprocess(query)


    query_text = " ".join(
        query_tokens
    )


    # Query -> TF-IDF Vector

    query_vector = search_vectorizer.transform(
        [query_text]
    )


    # Cosine Similarity

    similarity_scores = cosine_similarity(
        query_vector,
        search_matrix
    )[0]


    # Ranking

    ranked_indices = np.argsort(
        similarity_scores
    )[::-1]


    # Top-K Documents

    results = []


    for rank, idx in enumerate(
        ranked_indices[:top_k],
        start=1
    ):

        results.append({

            "Rank":
                rank,

            "Document ID":
                idx,

            "Similarity":
                round(
                    similarity_scores[idx],
                    4
                ),

            "Document preview":
                search_documents[idx][:200]

        })


    return pd.DataFrame(results)

## 10.3. Query examples

- medical image classification
- transformer language model
- deep learning healthcare
- natural language processing

In [49]:
# =====================================
# G3. Example Queries
# =====================================

queries = [

    "medical image classification",

    "transformer language model",

    "deep learning healthcare",

    "natural language processing"

]

## 10.4. Kết quả cần hiển thị

For each query display:

Rank | Document ID | Similarity | Document preview

In [50]:
# =====================================
# G4. Display Search Results
# =====================================

for query in queries:


    print("="*80)

    print("QUERY:",
          query)


    results = search_engine(
        query,
        top_k=5
    )


    display(results)

QUERY: medical image classification


,Rank,Document ID,Similarity,Document preview
0,1,18971,0.3162,the new rts <en env nvi vir iro ron onm nme me...
1,2,8527,0.2935,<hi his ist sto tor ory ry> of <ma mai aiz ize...
2,3,243,0.2675,j <he hea eal alt lth th> <po pol oli lit it> ...
3,4,28983,0.2658,q how do i <pu pur urc rch cha has ase se> <ba...
4,5,15873,0.2615,<th thi his is> one day <co cou our urs rse se...


QUERY: transformer language model


,Rank,Document ID,Similarity,Document preview
0,1,25428,0.3843,<no not ote te> if you re on an <ip iph pho ho...
1,2,24482,0.3656,<ha har ara ral ald ld> you are a co <ow own w...
2,3,13690,0.3629,<te tex ext xt> in <fi fin inn nni nis ish sh>...
3,4,4075,0.3567,<lo loo ook oki kin ing ng> for <sp spa pan an...
4,5,582,0.3314,<in int ntr tro rod odu duc uci cin ing ng> th...


QUERY: deep learning healthcare


,Rank,Document ID,Similarity,Document preview
0,1,9252,0.4207,san <di die ieg ego go> and <wa was ash shi hi...
1,2,9229,0.4114,<in inf nfl flu lue uen enc nce ce> <he hea ea...
2,3,8614,0.3791,as a <pa pat ati tie ien ent nt> you <ha hav a...
3,4,6123,0.3536,<wi wit ith th> <to tod oda day ay> s <ad adv ...
4,5,16272,0.3436,<de dem emi mi> <ra rad ade dev eva va> has a ...


QUERY: natural language processing


,Rank,Document ID,Similarity,Document preview
0,1,25428,0.3804,<no not ote te> if you re on an <ip iph pho ho...
1,2,13690,0.3771,<te tex ext xt> in <fi fin inn nni nis ish sh>...
2,3,24482,0.3522,<ha har ara ral ald ld> you are a co <ow own w...
3,4,4075,0.3488,<lo loo ook oki kin ing ng> for <sp spa pan an...
4,5,24136,0.3442,if you <wa wan ant nt> to <pi pic ick ck> a <p...


# 11. Part H — Evaluation

## 11.1. Tạo evaluation set

Prepare approximately 5–10 queries with student-provided relevance labels.

In [71]:
# =====================================
# H1. Evaluation relevance labels
# Based on Part G retrieved documents
# =====================================

EVAL_QUERIES = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
]


RELEVANCE_LABELS = {

    "medical image classification":
    {
        18971,
        8527,
        243
    },


    "transformer language model":
    {
        25428,
        24482,
        13690
    },


    "deep learning healthcare":
    {
        9252,
        9229,
        8614
    },


    "natural language processing":
    {
        25428,
        13690,
        24482
    }

}


assert set(EVAL_QUERIES) == set(RELEVANCE_LABELS)


print(
    "Evaluation queries:",
    len(EVAL_QUERIES)
)

print(
    "Queries manually judged:",
    sum(
        ids is not None
        for ids in RELEVANCE_LABELS.values()
    ),
    "/",
    len(EVAL_QUERIES)
)

Evaluation queries: 4
Queries manually judged: 4 / 4


## 11.2. Precision@K

P@K = number of relevant documents retrieved / K

In [72]:
# =====================================
# H2. Retrieve documents from search
# =====================================

def retrieve_documents(query, k=5):

    result = search_engine(
        query,
        top_k=k
    )

    return result["Document ID"].tolist()

In [73]:
# =====================================
# H3. Precision@5
# =====================================

def precision_at_k(
    retrieved_docs,
    relevant_docs,
    k=5
):

    retrieved_k = retrieved_docs[:k]


    relevant_retrieved = len(
        set(retrieved_k)
        &
        set(relevant_docs)
    )


    return relevant_retrieved / k

## 11.3. Recall@K

R@K = number of relevant documents retrieved / number of relevant documents

In [74]:
# =====================================
# H4. Recall@5
# =====================================

def recall_at_k(
    retrieved_docs,
    relevant_docs,
    k=5
):

    retrieved_k = retrieved_docs[:k]


    relevant_retrieved = len(
        set(retrieved_k)
        &
        set(relevant_docs)
    )


    return (
        relevant_retrieved
        /
        len(relevant_docs)
    )

## 11.4. Mean Reciprocal Rank

MRR = (1 / |Q|) × sum_q (1 / r_q)

In [75]:
# =====================================
# H5. Reciprocal Rank
# =====================================

def reciprocal_rank(
    retrieved_docs,
    relevant_docs
):

    for rank, doc_id in enumerate(
        retrieved_docs,
        start=1
    ):

        if doc_id in relevant_docs:
            return 1 / rank


    return 0

### Results export

Export only after the student supplies relevance labels.

In [76]:
# =====================================
# H6. Evaluation Results
# =====================================

evaluation_results = []


for query in EVAL_QUERIES:


    retrieved_docs = retrieve_documents(
        query,
        k=5
    )


    relevant_docs = RELEVANCE_LABELS[
        query
    ]


    p5 = precision_at_k(
        retrieved_docs,
        relevant_docs,
        k=5
    )


    r5 = recall_at_k(
        retrieved_docs,
        relevant_docs,
        k=5
    )


    rr = reciprocal_rank(
        retrieved_docs,
        relevant_docs
    )


    evaluation_results.append({

        "Query":
            query,

        "Retrieved":
            retrieved_docs,

        "Relevant":
            list(relevant_docs),

        "Precision@5":
            round(p5,4),

        "Recall@5":
            round(r5,4),

        "RR":
            round(rr,4)

    })


evaluation_table = pd.DataFrame(
    evaluation_results
)


evaluation_table

,Query,Retrieved,Relevant,Precision@5,Recall@5,RR
0,medical image classification,"[18971, 8527, 243, 28983, 15873]","[18971, 8527, 243]",0.6,1.0,1.0
1,transformer language model,"[25428, 24482, 13690, 4075, 582]","[24482, 25428, 13690]",0.6,1.0,1.0
2,deep learning healthcare,"[9252, 9229, 8614, 6123, 16272]","[9252, 9229, 8614]",0.6,1.0,1.0
3,natural language processing,"[25428, 13690, 24482, 4075, 24136]","[24482, 25428, 13690]",0.6,1.0,1.0


# 12. Part I — Error Analysis

Select 2 queries with good results and 2 queries with poor results.

In [ ]:
def inspect_query(query: str, model: dict, expected_relevant=None, top_k: int = 5):
    result_table = search(query, model, top_k=top_k)
    query_terms = set(model["preprocess"](query))
    overlaps = []
    for result in result_table.itertuples(index=False):
        index = model["id_to_index"][getattr(result, "Document_ID")]
        document_terms = set(model["tokenized_documents"][index])
        overlaps.append(len(query_terms & document_terms))
    if not result_table.empty:
        result_table = result_table.copy()
        result_table["Lexical overlap"] = overlaps
    return {
        "Query": query,
        "Expected relevant documents": None if expected_relevant is None else list(expected_relevant),
        "Retrieved documents": result_table,
    }

### Good query 1

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Good query 2

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Poor query 1

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

### Poor query 2

### Query

> TODO

### Expected relevant documents

> TODO

### Retrieved documents

> TODO: Run inspect_query() after labels and dataset are configured.

### Analysis

1. Vì sao document đứng đầu?

> TODO

2. Những từ nào đóng góp nhiều vào similarity?

> TODO

3. Có lexical overlap không?

> TODO

4. Có relevant document nào bị bỏ sót không?

> TODO

5. Failure xuất phát từ preprocessing, TF, IDF, vocabulary, lexical matching hay nguyên nhân khác?

> TODO

## Failure case quan trọng nhất

### Student analysis

> TODO

# 13. Part J — From Failure to the Next NLP Representation

TF-IDF
↓
Distributional representation
↓
Word Embedding
↓
Contextual Embedding
↓
Transformer

**Làm thế nào để biểu diễn được similarity về nghĩa thay vì chỉ similarity về từ?**

### Student hypothesis

> TODO: Đưa ra ít nhất một hypothesis.